# Performance

## Measurement Conditions and Scope

This page asks where execution time, memory and storage are spent. Charts,
supporting tables and observations are generated at book compilation from
the same document-reported aggregates as [Statistics](statistics.ipynb).
Source documents were reviewed on **9 November 2026**; analysis was revised
on **10 November 2026** without new benchmark runs.

- The documents describe Google Colab CPU-only execution. Exact configurations
  remain to be confirmed: the arXiv notebooks set three workers, while the
  document describes one active connection.
- arXiv values describe recorded Fast Crawl/Recovery invocations. Semantic
  Scholar values describe separate reruns dominated by saved caches.
- Stage timings with different runs/windows are not added to create an
  end-to-end measurement. RAM and disk claims also need a declared scope.

> **Full M1 benchmark: awaiting update P1--P4.** The
> [milestone requirements](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/)
> specify an auditable CPU-only Colab benchmark covering the complete system.
> The current stage results are useful partial evidence, not completion of
> that requirement. The checklist below identifies the missing measurements.

In [ ]:
from pathlib import Path
import sys

# Support execution from the project root or the notebook's own directory.
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "myst.yml").is_file() and (p / "milestone1" / "reporting.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from milestone1.reporting import load_results, total, integer, percent, seconds, hms, table

data = load_results()
arxiv = data["arxiv"]["ranges"]
s2 = data["semantic_scholar"]["ranges"]

from milestone1.reporting import analysis, COLORS, chart_style, finish_figure
from matplotlib.ticker import FuncFormatter

plt = chart_style()
labels = [f"Range {r['range']}" for r in arxiv]
positions = list(range(len(labels)))
comma = FuncFormatter(lambda value, _: f"{value:,.0f}")

## How Much Time Is Spent on Fast Crawl and Recovery?

**Figure 1.** Recorded arXiv phase durations in decimal hours. Each horizontal
bar represents a range; phase values are labeled inside, and their sum to the
right. Bar lengths use the printed phase times; labels are rounded to two
decimals. Recorded invocation time need not include calendar gaps between
Colab sessions. See the arithmetic audit below for rounding discrepancies.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 5.6), layout="constrained")
fast = [seconds(r["fast_hms"])/3600 for r in arxiv]
recovery = [seconds(r["recovery_hms"])/3600 for r in arxiv]
ax.barh(positions, fast, height=0.62, color=COLORS["blue"], label="Fast Crawl")
ax.barh(positions, recovery, left=fast, height=0.62,
        color=COLORS["orange"], label="Recovery")
for i, (f, r) in enumerate(zip(fast, recovery)):
    ax.text(f/2, i, f"{f:.2f}", ha="center", va="center",
            color="white", fontweight="bold", fontsize=13)
    ax.text(f+r/2, i, f"{r:.2f}", ha="center", va="center",
            color="#111827", fontweight="bold", fontsize=13)
    ax.text(f+r+0.3, i, f"{f+r:.2f} h", va="center",
            color="#111827", fontweight="bold", fontsize=12)
ax.set(yticks=positions, yticklabels=labels, xlim=(0, 22))
ax.set_xticks(range(0, 23, 2))
ax.invert_yaxis()
ax.set_xlabel("Recorded invocation time (hours)", fontsize=13, labelpad=10)
ax.set_title("arXiv: Fast Crawl and Recovery", fontsize=16, pad=16)
ax.tick_params(axis="both", labelsize=12)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0, pad=12)
ax.grid(axis="x", alpha=0.16)
fig.legend(*ax.get_legend_handles_labels(), loc="outside lower center",
           ncol=2, frameon=False, fontsize=13)
finish_figure(fig, "performance-arxiv-runtime",
              "Horizontal stacked bars show recorded Fast Crawl and Recovery hours for each range, with readable phase labels and totals.")

overall = data["arxiv"]["reported_overall"]
reported_time = seconds(overall["total_hms"])
assigned = total(arxiv, "assigned")
longest = max(arxiv, key=lambda r: seconds(r["total_hms"]))
shortest = min(arxiv, key=lambda r: seconds(r["total_hms"]))
analysis(
    f"The document reports {overall['total_hms']} of invocation time "
    f"({reported_time/3600:.2f} hours), equivalent to {reported_time/assigned:.3f} recorded "
    f"seconds per assigned ID. Recovery contributes "
    f"{percent(seconds(overall['recovery_hms']), reported_time, 1)}. "
    f"Range totals run from {shortest['total_hms']} (Range {shortest['range']}) to "
    f"{longest['total_hms']} (Range {longest['range']}).",
    "Recovery represents a substantial share of the recorded arXiv work. Its cost belongs in "
    "the reported stage time rather than being hidden behind the Fast Crawl measurement. "
    "The final failure count describes what remains unresolved; it is a different result from recovery effort.",
    "These are arXiv invocation ledgers as reported in the document, not full M1 timing or "
    "per-request latency. Initial failures and attempted recoveries are needed to quantify recovery "
    "effectiveness. Untracked or interrupted sessions can be absent; the printed totals also need the audit below."
)

> **Pending update P1:** original `runtime_history.csv` values in seconds,
> run boundaries and actual configurations to reconcile the printed times.
> **Pending update S1:** before/after Recovery records to measure successful
> recoveries and explain reasons for failure. Final error counts alone do
> not measure recovery effectiveness.

## What Do the Semantic Scholar Timers Include?

**Figure 2.** Accumulated fresh-lookup fetch timers and notebook-cell wall
times. Panels have different vertical scales because the timer scopes
differ; read their numerical values rather than comparing bar heights
across panels. Labels give the number of fresh publication lookups.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), layout="constrained")
fresh = [r["input_papers"]-r["cache_hits"]-r["invalid_id"] for r in s2]
tick_labels = [f"Range {r['range']}\n{n} fresh" for r,n in zip(s2,fresh)]
for ax, key, title, color in (
    (axes[0], "fetch_seconds", "Accumulated fresh-lookup fetch time", COLORS["blue"]),
    (axes[1], "cell_seconds", "Notebook-cell wall time", COLORS["purple"]),
):
    values = [r[key] for r in s2]
    bars = ax.bar(positions, values, color=color)
    ax.bar_label(bars, labels=[f"{v:,.2f}" for v in values], padding=4, fontsize=9)
    ax.set(xticks=positions, xticklabels=tick_labels, ylim=(0, max(values)*1.2),
           ylabel="Time (seconds)", title=title)
    ax.grid(axis="y", alpha=0.18)
finish_figure(fig, "performance-semantic-scholar-timing",
              "Cache-heavy Semantic Scholar rerun timings with separate scales for fetch and cell time, and fresh lookup counts.")

cached_count = total(s2, "cache_hits")
inputs = total(s2, "input_papers")
fresh_count = inputs-cached_count-total(s2, "invalid_id")
largest_cell = max(s2, key=lambda r: r["cell_seconds"])
analysis(
    f"Across the five reruns, {integer(cached_count)} / {integer(inputs)} inputs are cache hits "
    f"({percent(cached_count, inputs, 3)}), leaving {integer(fresh_count)} fresh lookups. "
    f"Range {largest_cell['range']} reports {largest_cell['fetch_seconds']:.2f} seconds of "
    f"accumulated fetch time and {largest_cell['cell_seconds']:,.2f} seconds of cell wall time.",
    "The fetch timer and the cell timer answer different questions. Fresh lookup timers stop "
    "before normalization/cache writes; cell time includes local processing and cache reads, "
    "but excludes earlier checkpoint loading and later table export/validation. Their difference "
    "highlights why timer boundaries must accompany a result.",
    "These cache-heavy reruns do not measure fresh retrieval of the entire corpus. The aggregates "
    "do not identify the cause of the fetch/cell gap or support an API-speed ranking of ranges. "
    "A lookup may include multiple HTTP attempts, and the mean-fetch field is per fresh publication, not per attempt."
)

> **Pending update P2:** original crawl logs or an explicitly scoped
> additional benchmark, including checkpoint loading, reference collection,
> normalization and export. Cache state must accompany each measurement.
> The current cell/fetch timers do not cover all of those steps.

## What Memory Was Observed in the Reference Rerun?

**Figure 3.** Average and peak sampled Semantic Scholar notebook-process
RSS, in MiB. The two series use one zero-based scale and solid colors;
nearby labels retain the exact reported values without a truncated axis.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 5.8), layout="constrained")
width = 0.32
averages = [r["average_rss_mib"] for r in s2]
peaks = [r["peak_rss_mib"] for r in s2]
avg_bars = ax.barh([i-width/2 for i in positions], averages, height=width,
                   color=COLORS["blue"], label="Average sampled RSS")
peak_bars = ax.barh([i+width/2 for i in positions], peaks, height=width,
                    color=COLORS["orange"], label="Peak sampled RSS")
for bars, values in ((avg_bars, averages), (peak_bars, peaks)):
    ax.bar_label(bars, labels=[f"{v:.2f}" for v in values], padding=5, fontsize=10)
ax.set(yticks=positions, yticklabels=labels, xlim=(0, 330),
       xlabel="Notebook-process RSS (MiB)", title="Semantic Scholar: cache-heavy rerun memory")
ax.invert_yaxis()
ax.tick_params(labelsize=12)
ax.grid(axis="x", alpha=0.16)
fig.legend(*ax.get_legend_handles_labels(), loc="outside lower center", ncol=2, frameon=False)
finish_figure(fig, "performance-semantic-scholar-memory",
              "Horizontal bars compare average and peak notebook-process RSS on a shared zero-based MiB axis.")

analysis(
    f"Reported sampled peaks range from {min(peaks):.2f} to {max(peaks):.2f} MiB. "
    f"The maximum difference between the average and peak within a range is "
    f"{max(b-a for a,b in zip(averages,peaks)):.2f} MiB.",
    "These measurements describe sampled notebook-process RSS during the reported cache-heavy "
    "crawl cell, including existing kernel memory. The shared zero-based scale shows the "
    "near-equality of average and peak without exaggerating small differences.",
    "Sampling every 0.5 seconds may miss brief peaks. These values do not include every M1 stage "
    "or establish fresh-crawl memory requirements. A pooled RAM mean needs compatible sampling "
    "windows/weights; the five range means are not simply averaged into a full-system result."
)

> **Pending update P3:** original RSS samples and measurement windows for
> both stages, plus full M1 peak and average RAM on the same execution
> interval. Include any relevant subprocesses and state the memory definition.
> Do not sum stage peaks observed at unrelated times.

## How Much Persistent Storage Do the arXiv Checkpoints Use?

**Figure 4.** Final compressed arXiv checkpoint sizes, including retained
source files and bookkeeping. Values are in MiB; this chart describes
persistent stage output rather than transient/peak disk consumption.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.4), layout="constrained")
sizes = [r["checkpoint_mib"] for r in arxiv]
bars = ax.bar(positions, sizes, color=COLORS["teal"])
ax.bar_label(bars, labels=[f"{v:,.2f}" for v in sizes], padding=4)
ax.set(xticks=positions, xticklabels=labels, ylim=(0, 350),
       ylabel="Compressed checkpoint storage (MiB)", title="arXiv: retained checkpoint storage")
ax.grid(axis="y", alpha=0.18)
finish_figure(fig, "performance-checkpoint-storage",
              "Final compressed arXiv checkpoint sizes range from 237.15 to 291.45 MiB.")

size = total(arxiv, "checkpoint_mib")
smallest = min(arxiv, key=lambda r: r["checkpoint_mib"])
largest = max(arxiv, key=lambda r: r["checkpoint_mib"])
analysis(
    f"The retained arXiv checkpoints total {size:,.2f} MiB ({size/1024:.2f} GiB). "
    f"Range sizes run from {smallest['checkpoint_mib']:.2f} MiB (Range {smallest['range']}) "
    f"to {largest['checkpoint_mib']:.2f} MiB (Range {largest['range']}).",
    "Compressed checkpoint size describes the persistent arXiv stage output, including retained "
    "sources and bookkeeping. Equal assigned ID counts do not imply equal source volume; "
    "version counts and source composition require record-level inspection to explain the difference.",
    "This is neither peak disk during execution nor the complete M1 output including Semantic Scholar. "
    "It is also not the required mean source size before/after figure removal; that comparison "
    "needs paired source measurements on a consistent basis (Statistics, pending S2)."
)

> **Pending update P4:** peak disk throughout the full workflow, including
> working/temporary/checkpoint/output directories, and final output bytes
> for the complete M1 corpus including Semantic Scholar. Report compression,
> storage locations and exclusions, and avoid counting the same files twice.

## Required Measurements Awaiting Update

**Table 1.** Current coverage of the remaining performance measurements.
Status describes available evidence, not whether the code has fields for a
metric. Missing measurements remain unavailable and are not plotted as zero.

In [ ]:
table(["Item", "Available evidence", "Update needed"], [
    ["P1: conditions and timing audit", "Reported CPU-only Colab environment; conflicting worker descriptions",
     "Actual code/configuration, range/run IDs, cache state and unrounded ledgers"],
    ["P2: full-system wall time", "Separate arXiv invocation and reference-cell timers",
     "Common start before discovery and end after completed M1 outputs; auditable run boundaries"],
    ["P2: per-paper time and discovery", "arXiv recorded seconds per assigned ID; batch discovery fields exist",
     "Total discovery time, defined processing denominator and per-paper timers where recorded"],
    ["P3: peak and average RAM", "Reference-rerun notebook RSS only",
     "Full-window samples for M1 processes, sampling interval and aggregation rule"],
    ["P4: peak disk", "No complete full-system peak supplied",
     "Measured maximum including temporary files and packaging over the entire workflow"],
    ["P4: final M1 output", "Compressed arXiv checkpoint sizes",
     "Reference cache/table output bytes and complete final total with counting/compression scope"],
])

The group must agree on one coordinated benchmark scope and owner. Existing
logs should be examined first; any still-unmeasured required items need an
appropriate additional measurement before the benchmark can be described
as complete. A cache-heavy rerun must remain labeled as such.

Store timer boundaries, input IDs/ranges, code/configuration and resource
samples with the results. Full-system peak RAM is based on concurrent memory
use within the declared window; average RAM needs compatible time/sample
weighting. Disk tracking must continue through packaging/export. Fields in
a latest-batch checkpoint can refer to Recovery rather than all prior work.

Publication-level time/size distributions and medians/quantiles can supplement
the required means when records become available (**additional analysis S4**).
A range average is not an observed per-paper distribution. Paired source-size
means remain a separate required statistic (**S2**, on the Statistics page).

## Supporting Timing Audit

**Table 2.** Differences between the document's printed arXiv times and
sums of those printed values. Originals are retained pending unrounded
ledgers. No reported benchmark is silently replaced by a rounded sum.

In [ ]:
overall = data["arxiv"]["reported_overall"]
audit = []
for r in arxiv:
    calculated = seconds(r["fast_hms"]) + seconds(r["recovery_hms"])
    reported = seconds(r["total_hms"])
    if calculated != reported:
        audit.append([f"Range {r['range']}: phase sum", r["total_hms"], hms(calculated), f"{calculated - reported:+d}"])
for label, key in (("Overall Fast Crawl: row sum", "fast_hms"),
                   ("Overall Recovery: row sum", "recovery_hms"),
                   ("Overall total: row sum", "total_hms")):
    calculated = sum(seconds(r[key]) for r in arxiv)
    reported = seconds(overall[key])
    audit.append([label, overall[key], hms(calculated), f"{calculated - reported:+d}"])
phase_sum = sum(seconds(r["fast_hms"]) + seconds(r["recovery_hms"]) for r in arxiv)
audit.append(["Overall: sum of all printed phases", overall["total_hms"], hms(phase_sum),
              f"{phase_sum - seconds(overall['total_hms']):+d}"])
table(["Check", "Reported", "Recomputed from printed values", "Difference (s)"], audit)

These small arithmetic differences require reconciliation; they do not
explain the larger inter-range runtime differences. Original timing ledgers,
`metrics.csv`, checkpoint metrics, reference-run metrics and resource samples
are the evidence for the next update. Neither a fresh crawl nor an external
API request is performed while compiling this report.